# Assignment 1 — Credit Card Default: EDA & Classification

**Course:** Machine Learning (2026W)  
**Author:** Will Sutherland  
**Dataset:** [UCI Credit Card Default](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) — 30,000 Taiwanese credit card holders, binary default prediction

---

## Overview

This notebook explores a classic credit risk classification problem and compares two supervised learning approaches — **Random Forest** and **K-Nearest Neighbours (KNN)** — using a full scikit-learn pipeline with cross-validated hyperparameter tuning.

**Key topics covered:**
- Exploratory data analysis (EDA) and class imbalance diagnosis
- Preprocessing pipeline (imputation, scaling, one-hot encoding) with `ColumnTransformer`
- Hyperparameter tuning via `GridSearchCV` with 5-fold stratified CV
- Model comparison using ROC-AUC
- Threshold selection via Youden's J statistic and its accuracy trade-off


## 0 · Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# ── Reproducibility & display ──────────────────────────────────────────────
SEED = 123
np.random.seed(SEED)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# ── Data source ─────────────────────────────────────────────────────────────
# Dataset is publicly available from the UCI ML Repository.
# Download and update the path below, or load directly from the URL.
DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/"
    "default%20of%20credit%20card%20clients.xls"
)


## 1 · Exploratory Data Analysis

We examine the structure and distributional characteristics of the dataset before modelling.  
Key findings:
- **30,000 rows × 25 columns**; all features imported as numeric (including categorical ones like `SEX`, `EDUCATION`, `MARRIAGE`)
- **No missing values**
- Target is mildly imbalanced: **77.9% non-default, 22.1% default**
- Financial features (`BILL_AMT*`, `PAY_AMT*`) exhibit strong right skew, suggesting large outliers


In [ ]:
# ── Load & clean ───────────────────────────────────────────────────────────
df = pd.read_excel(DATA_URL, sheet_name="Data", skiprows=1)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df = df.rename(columns={"default_payment_next_month": "default"})

print("Shape:", df.shape)
display(df.head())
display(df.describe())


In [ ]:
def data_overview(df: pd.DataFrame) -> pd.DataFrame:
    """Return dtype, missing count, and unique count for each column."""
    return pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "missing":  df.isna().sum(),
        "n_unique": df.nunique(dropna=False),
    })

display(data_overview(df))


In [ ]:
# ── Target distribution ────────────────────────────────────────────────────
ax = sns.countplot(
    data=df, x="default", hue="default",
    palette={0: "steelblue", 1: "firebrick"},
    edgecolor="black", linewidth=1.2, legend=False
)
ax.set_title("Target Variable Distribution", fontsize=13)
ax.set_xlabel("Default (1 = Yes)")
ax.set_ylabel("Count")

total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    ax.annotate(f"{count:,} ({100*count/total:.1f}%)",
                (p.get_x() + p.get_width() / 2., count),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ── Univariate distributions (selected features) ───────────────────────────
selected_cols = ["limit_bal", "age", "pay_0", "bill_amt1", "pay_amt1"]
df[selected_cols].hist(bins=30, figsize=(14, 8), edgecolor="black")
plt.suptitle("Univariate Distributions — Selected Financial & Demographic Features", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Pair plot coloured by default status ───────────────────────────────────
sns.pairplot(df[selected_cols + ["default"]], hue="default", plot_kws={"alpha": 0.3, "s": 10})
plt.suptitle("Pair Plot — Selected Features by Default Status", y=1.02)
plt.show()


## 2 · Preprocessing Pipeline

The preprocessing is implemented entirely within a `Pipeline` to **prevent data leakage** — all fit operations happen on training data only and are then applied to the test set.

- **Numeric features:** median imputation → `StandardScaler` (important for KNN, which is distance-based)  
- **Categorical features:** mode imputation → `OneHotEncoder`  
- Although no missing values were found, imputers are included for production robustness.


In [ ]:
TARGET = "default"

X = df.drop(columns=["id", TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.3f}  |  Test default rate: {y_test.mean():.3f}")


In [ ]:
cat_cols = ["sex", "education", "marriage"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe,    num_cols),
    ("cat", categorical_pipe, cat_cols),
], remainder="drop")


## 3 · Model Training & Hyperparameter Tuning

### 3.1 — Helper Functions

In [ ]:
def plot_gridsearch_curve(gs, param_name: str, title: str) -> pd.DataFrame:
    """Plot mean CV ROC-AUC vs the tuned hyperparameter."""
    results = pd.DataFrame(gs.cv_results_)
    x = results[f"param_{param_name}"].astype(int)
    y_score = results["mean_test_score"]
    plt.figure()
    plt.plot(x, y_score, marker="o")
    plt.xlabel(param_name.replace("model__", ""))
    plt.ylabel("Mean CV ROC-AUC")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    return results


def plot_roc_and_choose_threshold(model, X, y, title: str, rule: str = "youden") -> float:
    """
    Plot the ROC curve and return a recommended decision threshold.
    rule='youden' selects the threshold maximising (TPR - FPR).
    """
    y_prob = model.predict_proba(X)[:, 1]
    fpr, tpr, thresholds = roc_curve(y, y_prob)
    auc = roc_auc_score(y, y_prob)

    if rule == "youden":
        idx = (tpr - fpr).argmax()
        best_thr = thresholds[idx]

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    plt.scatter(fpr[idx], tpr[idx], color="red", zorder=5,
                label=f"Threshold = {best_thr:.3f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()
    return best_thr


### 3.2 — Random Forest

We tune `n_estimators` via 5-fold CV with ROC-AUC as the scoring metric. Performance stabilises as the forest grows, plateauing around 50 trees.

In [ ]:
rf_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model",      RandomForestClassifier(random_state=SEED))
])

rf_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid={"model__n_estimators": [4, 5, 10, 20, 50]},
    cv=5, scoring="roc_auc", n_jobs=-1
)
rf_search.fit(X_train, y_train)

print(f"Best params: {rf_search.best_params_}")
print(f"Best CV ROC-AUC: {rf_search.best_score_:.4f}")

rf_results = plot_gridsearch_curve(
    rf_search, "model__n_estimators",
    "Random Forest: Mean CV ROC-AUC vs n_estimators"
)


### 3.3 — K-Nearest Neighbours

We tune `n_neighbors`. Smaller k values overfit; larger k values produce smoother, more generalizable decision boundaries.

In [ ]:
knn_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model",      KNeighborsClassifier())
])

knn_search = GridSearchCV(
    estimator=knn_pipe,
    param_grid={"model__n_neighbors": [3, 5, 10, 20]},
    cv=5, scoring="roc_auc", n_jobs=-1
)
knn_search.fit(X_train, y_train)

print(f"Best params: {knn_search.best_params_}")
print(f"Best CV ROC-AUC: {knn_search.best_score_:.4f}")

_ = plot_gridsearch_curve(
    knn_search, "model__n_neighbors",
    "KNN: Mean CV ROC-AUC vs n_neighbors"
)


## 4 · Model Comparison & Threshold Selection

In [ ]:
# ── Side-by-side CV performance ────────────────────────────────────────────
cv_compare = pd.DataFrame({
    "Model":          ["Random Forest", "KNN"],
    "Best CV ROC-AUC": [rf_search.best_score_, knn_search.best_score_],
    "Best Params":    [rf_search.best_params_, knn_search.best_params_],
}).sort_values("Best CV ROC-AUC", ascending=False)
display(cv_compare)


In [ ]:
best_rf  = rf_search.best_estimator_
best_knn = knn_search.best_estimator_

rf_thr  = plot_roc_and_choose_threshold(best_rf,  X_test, y_test, "Random Forest — Test ROC")
knn_thr = plot_roc_and_choose_threshold(best_knn, X_test, y_test, "KNN — Test ROC")

print(f"Recommended threshold (Youden's J):  RF = {rf_thr:.3f}  |  KNN = {knn_thr:.3f}")


In [ ]:
def compare_thresholds(model, X, y, thr, name):
    """Print accuracy at the default 0.5 threshold vs the Youden-optimal one."""
    y_prob = model.predict_proba(X)[:, 1]
    acc_default = accuracy_score(y, (y_prob >= 0.50).astype(int))
    acc_optimal = accuracy_score(y, (y_prob >= thr).astype(int))
    print(f"\n{name}")
    print(f"  Accuracy @ 0.50 threshold : {acc_default:.4f}")
    print(f"  Accuracy @ {thr:.3f} threshold : {acc_optimal:.4f}")

compare_thresholds(best_rf,  X_test, y_test, rf_thr,  "Random Forest")
compare_thresholds(best_knn, X_test, y_test, knn_thr, "KNN")


## 5 · Summary & Key Takeaways

| Model | Best CV ROC-AUC | Best Params |
|---|---|---|
| Random Forest | **0.7579** | n_estimators = 50 |
| KNN | 0.7440 | n_neighbors = 20 |

**Random Forest outperforms KNN** on this dataset. The performance gap is moderate but consistent across cross-validation folds.

**Threshold selection matters in imbalanced classification.** The default 0.5 threshold favours the majority class. Youden's J lowers the threshold (≈0.285 for RF), trading overall accuracy (~81.6% → ~76.8%) for substantially higher sensitivity — catching more defaulters, which is the metric that matters in credit risk.

**Preprocessing inside the pipeline is non-negotiable.** Fitting the scaler on the full dataset before splitting would leak test-set statistics into training — a common but serious mistake that inflates estimated performance.
